In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

OUTPUT_FILE = "code_subject.csv"
URL = "https://www.ilovephd.com/list-of-subject-areas-covered-by-scopus-database/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

CODE_MAP = {
    "10": "Multidisciplinary",
    "11": "Agri & Bio",
    "12": "Arts & Hum",
    "13": "Biochem & Genetics",
    "14": "Business & Mgmt",
    "15": "Chem Eng",
    "16": "Chemistry",
    "17": "Comp Sci",
    "18": "Decision Sci",
    "19": "Earth & Planet",
    "20": "Econ & Finance",
    "21": "Energy",
    "22": "Engineering",
    "23": "Environment",
    "24": "Immuno & Micro",
    "25": "Materials",
    "26": "Math",
    "27": "Medicine",
    "28": "Neuroscience",
    "29": "Nursing",
    "30": "Pharma & Tox",
    "31": "Physics",
    "32": "Psychology",
    "33": "Social Sci",
    "34": "Vet Med",
    "35": "Dentistry",
    "36": "Health"
}

def get_scopus_data():
    print(f"Fetching data from {URL}...")
    try:
        response = requests.get(URL, headers=HEADERS, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        figure = soup.find('figure', class_='wp-block-table')

        if not figure:
            print("No table found on page.")
            return None

        rows = figure.find('table').find_all('tr')
        data_list = []

        seen_codes = set()

        for row in rows:
            cols = row.find_all('td')
            if len(cols) >= 2:
                raw_code_text = cols[0].text.strip()

                if len(raw_code_text) >= 2 and raw_code_text[:2].isdigit():

                    prefix = raw_code_text[:2]

                    short_name = CODE_MAP.get(prefix, "Other")

                    if prefix not in seen_codes:
                        data_list.append({
                            "code": int(prefix),
                            "name": short_name
                        })
                        seen_codes.add(prefix)

        return pd.DataFrame(data_list)

    except Exception as e:
        print(f"Error: {e}")
        return None

if __name__ == "__main__":
    df = get_scopus_data()

    if df is not None and not df.empty:
        df = df.sort_values(by="code")
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"Success! Saved {len(df)} main topics to {OUTPUT_FILE}")
    else:
        print("No data extracted.")

Fetching data from https://www.ilovephd.com/list-of-subject-areas-covered-by-scopus-database/...
Success! Saved 27 main topics to code_subject.csv
